# From Pixels to Provenance - Stage 0 to Stage 2c (review-corrected)

Phases 0-2 of the proposal pipeline. Changes made from the review:

**Stage 1c (critical)** - now anchors a single **detection band**: it runs Background2D -> detect_sources -> deblend_sources to build a master segmentation map and source list, cuts every filter at the same RA/Dec via absolute WCS, and applies **segmentation-based neighbour masking** (companion-source pixels zeroed) so close neighbours cannot corrupt the Stage 1e / statmorph features.

**Stage 1e (improvement)** - `m20_index` and `sersic_n` are now **real deterministic measurements** (Lotz-style M20; a bounded 1D Sersic-profile fit), and `flag_sersic` is emitted, so every feature column is populated and the tabular width is stable across the NumPy fallback and the live statmorph path.

**Stage 2a (improvement + confirmation)** - the imputer already fit only when `fit=True` (no validation/test leakage - the review's pasted fix matched the existing line). Added `SimpleImputer(keep_empty_features=True)` with a **fixed feature schema** so the fused matrix is the same width whether `m20`/`sersic_n` are NaN (fallback) or filled (statmorph).

**Stage 2b (improvement + confirmation)** - `prediction_data=True` was already set for the contrib HDBSCAN (the review's pasted fix matched the existing line). Added `n_init=10` to the GMM so the BIC sweep uses multiple restarts and avoids local minima.

Gaia remains in **Stage 2c** (baseline segregation), not 2b. Stage 2d onward is not coded yet.


In [ ]:
# ============================================================
# Stage 0 — Data Governance: spatial split with a leakage collar
# ============================================================
import json
from typing import Dict, Any, Optional
import numpy as np


class SkyRegionPartitioner:
    """
    Establishes reproducible train / dev / external splits over sky regions
    and PREVENTS spatial leakage by reserving a buffer collar around every
    region. Sources that fall inside a collar (or overlap two regions) are
    dropped, not assigned. The RA buffer is cos(dec)-corrected so the collar
    is a true on-sky angular width at all declinations.
    """

    def __init__(self, output_path: str = "split_rule.json", buffer_arcmin: float = 2.0):
        self.output_path = output_path
        self.buffer_deg = buffer_arcmin / 60.0
        self.registry: Dict[str, Any] = {}

    def define_region(self, name: str, ra_min: float, ra_max: float,
                      dec_min: float, dec_max: float, split_tag: str) -> None:
        self.registry[name] = {
            "split_tag": split_tag,
            "ra_range": [ra_min, ra_max],
            "dec_range": [dec_min, dec_max],
        }

    def assign_split(self, ra: float, dec: float) -> Optional[str]:
        """Return the split tag for a source, or None if it must be excluded
        (inside a collar, or ambiguous between two regions)."""
        inner_hit = None
        for reg in self.registry.values():
            r0, r1 = reg["ra_range"]
            d0, d1 = reg["dec_range"]
            cosd = np.cos(np.deg2rad(0.5 * (d0 + d1)))
            b_ra = self.buffer_deg / max(cosd, 1e-3)   # cos(dec) correction
            in_inner = (r0 <= ra <= r1) and (d0 <= dec <= d1)
            in_outer = (r0 - b_ra <= ra <= r1 + b_ra) and \
                       (d0 - self.buffer_deg <= dec <= d1 + self.buffer_deg)
            if in_inner:
                if inner_hit is not None:
                    return None                        # overlaps two inner regions
                inner_hit = reg["split_tag"]
            elif in_outer:
                return None                            # inside protective collar
        return inner_hit

    def apply_to_catalogue(self, ra_arr, dec_arr):
        """Vectorised convenience wrapper returning a split tag per source."""
        return [self.assign_split(float(r), float(d)) for r, d in zip(ra_arr, dec_arr)]

    def freeze_ruleset(self) -> str:
        payload = {"buffer_deg": self.buffer_deg, "regions": self.registry}
        with open(self.output_path, "w") as f:
            json.dump(payload, f, indent=4)
        print(f"[Stage 0] Ruleset frozen -> {self.output_path}")
        return self.output_path


# --- Real CANDELS/COSMOS field footprints ---
partitioner = SkyRegionPartitioner(buffer_arcmin=2.0)
partitioner.define_region("COSMOS_CORE", 149.7, 150.5, 1.9, 2.7, split_tag="train")
partitioner.define_region("GOODS_N_SUB", 188.9, 189.4, 62.1, 62.4, split_tag="dev")
partitioner.define_region("CANDELS_EGS", 214.6, 215.3, 52.6, 53.1, split_tag="external")
rulebook_path = partitioner.freeze_ruleset()

# Demonstrate the collar actually excludes near-boundary sources:
for ra, dec in [(150.0, 2.30), (150.52, 2.30), (214.90, 52.80), (0.0, 0.0)]:
    print(f"  ({ra:8.3f}, {dec:6.2f}) -> {partitioner.assign_split(ra, dec)}")


In [ ]:
# ============================================================
# Stage 1a — MAST acquisition, driven by the Stage-0 regions
# ============================================================
import os, json
import numpy as np
import pandas as pd

try:
    from astropy.io import fits
    from astroquery.mast import Observations
    _HAVE_MAST = True
except Exception as e:
    _HAVE_MAST = False
    print(f"[Stage 1a] astroquery/astropy unavailable ({e}); scaffold only.")


class ArchiveAcquisitionPipeline:
    """
    Queries MAST by REGION (matching Stage 0) for Level-3, imaging, SCIENCE
    products on the instruments/filters the downstream cutout + PSF stages
    expect, then batch-downloads and structurally verifies each FITS file.
    """

    def __init__(self, download_dir: str = "./cache",
                 instruments=("ACS/WFC", "WFC3/UVIS"),
                 filters=("F606W", "F814W")):
        self.download_dir = download_dir
        self.instruments = list(instruments)
        self.filters = list(filters)
        os.makedirs(self.download_dir, exist_ok=True)
        self.manifest_path = os.path.join(self.download_dir, "manifest.json")

    def harvest_region(self, ra_min, ra_max, dec_min, dec_max) -> pd.DataFrame:
        """Cone/box query on the region centre; keep Level-3 imaging DRC/DRZ."""
        if not _HAVE_MAST:
            return pd.DataFrame()
        ra_c, dec_c = 0.5 * (ra_min + ra_max), 0.5 * (dec_min + dec_max)
        radius = np.hypot(ra_max - ra_min, dec_max - dec_min) / 2.0
        print(f"[Stage 1a] Querying MAST around ({ra_c:.3f}, {dec_c:.3f}), r={radius:.3f} deg")

        obs = Observations.query_criteria(
            coordinates=f"{ra_c} {dec_c}", radius=f"{radius} deg",
            obs_collection="HST", dataproduct_type="image",
            calib_level=3, instrument_name=self.instruments,
            filters=self.filters,
        )
        if len(obs) == 0:
            return pd.DataFrame()

        prod = Observations.get_product_list(obs)
        prod = Observations.filter_products(
            prod, productType="SCIENCE",
            productSubGroupDescription=["DRC", "DRZ"],
        )
        return prod.to_pandas()

    def secure_science_products(self, product_df: pd.DataFrame, max_files: int = 5) -> None:
        if not _HAVE_MAST or product_df.empty:
            print("[Stage 1a] Nothing to download.")
            return

        # Prefer DRC over DRZ, then batch-download in ONE call (astroquery batches).
        product_df = product_df.copy()
        product_df["prio"] = product_df["productFilename"].str.endswith("_drc.fits").astype(int)
        product_df = product_df.sort_values("prio", ascending=False).head(max_files)

        from astropy.table import Table
        manifest_tbl = Observations.download_products(
            Table.from_pandas(product_df.drop(columns=["prio"])),
            download_dir=self.download_dir,
        )

        manifest = {}
        for row in manifest_tbl:
            if str(row["Status"]).lower() != "completed":
                continue
            local_path = row["Local Path"]          # trust astroquery's real path
            fname = os.path.basename(local_path)
            try:
                with fits.open(local_path) as hdul:
                    has_sci = "SCI" in hdul
                    wcs_ok = has_sci and hdul["SCI"].header.get("CTYPE1", "").startswith("RA")
                if has_sci and wcs_ok:
                    manifest[fname] = {
                        "local_path": local_path,
                        "product_type": "drc" if fname.endswith("_drc.fits") else "drz",
                        "integrity_verified": True,
                    }
            except Exception as e:
                print(f"[Stage 1a] Structural check failed for {fname}: {e}")

        with open(self.manifest_path, "w") as f:
            json.dump(manifest, f, indent=4)
        print(f"[Stage 1a] Verified {len(manifest)} product(s); manifest -> {self.manifest_path}")


# Example: acquire from the Stage-0 TRAIN region (COSMOS), not an unrelated target.
acq = ArchiveAcquisitionPipeline()
prods = acq.harvest_region(149.7, 150.5, 1.9, 2.7)
if not prods.empty:
    acq.secure_science_products(prods, max_files=2)
else:
    print("[Stage 1a] (Skipped live download — running without MAST access.)")


In [ ]:
# ============================================================
# Stage 1b — QC autoencoder: UNDER-COMPLETE bottleneck + validated gate
# ============================================================
# Methodology (from the ISSM "Data-driven & representation learning" deck):
#   - Convolutional generative autoencoder trained on ||x - x_hat||^2  (slides 25, 27, 29)
#   - Reconstruction-error anomaly detection: artifacts are rare, so the model
#     never learns them and they reconstruct badly  (slides 37, 42)
#   - Validation = train on CLEAN tiles, plot recon-error on held-out data,
#     set the threshold from that distribution  (slide 43)
#   - The FC bottleneck makes the AE UNDER-complete (slide 28): 128x128 = 16384 px
#     is squeezed to latent_dim << 16384, so a clean tile reconstructs well and an
#     artifact does not. The SAME latent feeds Stage 1f, so QC and clustering
#     share one representation.
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    _HAVE_TORCH = True
except Exception as e:
    _HAVE_TORCH = False
    print(f"[Stage 1b] torch unavailable ({e}); scaffold only.")

try:
    import matplotlib.pyplot as plt
    _HAVE_PLT = True
except Exception:
    _HAVE_PLT = False

try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    _HAVE_SK = True
except Exception:
    _HAVE_SK = False


def asinh_normalise(patch, q_lo=1.0, q_hi=99.5, soft=0.1):
    """Percentile-clipped asinh stretch - robust to hot pixels / cosmic rays,
    unlike raw min-max (a single hot pixel pins the max at 1)."""
    p = patch.astype(np.float32)
    lo, hi = np.percentile(p, [q_lo, q_hi])
    x = np.clip((p - lo) / max(hi - lo, 1e-8), 0, 1)
    return (np.arcsinh(x / soft) / np.arcsinh(1.0 / soft)).astype(np.float32)


if _HAVE_TORCH:
    class QualityControlCAE(nn.Module):
        """Under-complete convolutional autoencoder. Conv stack 128x128 -> 64x16x16,
        then a Linear bottleneck to latent_dim (<< 16384) and back. forward() returns
        (reconstruction, latent); encode() returns the latent Stage 1f consumes."""

        def __init__(self, latent_dim: int = 128):
            super().__init__()
            self.encoder_conv = nn.Sequential(
                nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),   # 128 -> 64
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),  # 64 -> 32
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),  # 32 -> 16
            )
            self._feat = 64 * 16 * 16                                  # 16384
            self.enc_fc = nn.Linear(self._feat, latent_dim)           # UNDER-complete
            self.dec_fc = nn.Linear(latent_dim, self._feat)
            self.decoder_conv = nn.Sequential(
                nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),   # 16 -> 32
                nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),   # 32 -> 64
                nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1), nn.Sigmoid(), # 64 -> 128
            )

        def encode(self, x):
            h = self.encoder_conv(x).flatten(1)
            return self.enc_fc(h)                                      # [N, latent_dim]

        def decode(self, z):
            h = self.dec_fc(z).view(-1, 64, 16, 16)
            return self.decoder_conv(h)

        def forward(self, x):
            z = self.encode(x)
            return self.decode(z), z

    def _to_tensor(tiles):
        arr = np.stack([asinh_normalise(t) for t in tiles])
        return torch.tensor(arr).unsqueeze(1)                         # [N,1,128,128]

    def train_qc(model, clean_tiles, epochs=20, batch_size=64,
                 lr=1e-3, weight_decay=1e-5, device="cpu", verbose=True):
        """Train ONLY on clean tiles (slide 42 rarity assumption). Optimiser recipe
        from slide 29: MSELoss + Adam(lr=1e-3, weight_decay=1e-5)."""
        model.to(device).train()
        loader = DataLoader(TensorDataset(_to_tensor(clean_tiles)),
                            batch_size=batch_size, shuffle=True)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        loss_fn = nn.MSELoss()
        for ep in range(epochs):
            running = 0.0
            for (xb,) in loader:
                xb = xb.to(device)
                recon, _ = model(xb)
                loss = loss_fn(recon, xb)
                opt.zero_grad(); loss.backward(); opt.step()
                running += loss.item() * xb.size(0)
            if verbose:
                print(f"  epoch {ep + 1:2d}/{epochs}  train MSE = {running / len(loader.dataset):.5f}")
        return model

    @torch.no_grad()
    def recon_errors(model, tiles, device="cpu"):
        model.to(device).eval()
        x = _to_tensor(tiles).to(device)
        r, _ = model(x)
        return torch.mean((x - r) ** 2, dim=(1, 2, 3)).cpu().numpy()

    def calibrate_threshold(model, val_clean_tiles, percentile=99.0, plot=True):
        """slide 43: run the trained model on held-out CLEAN tiles, inspect the
        reconstruction-error distribution, set the pass/fail threshold from it."""
        errs = recon_errors(model, val_clean_tiles)
        thr = float(np.percentile(errs, percentile))
        if plot and _HAVE_PLT:
            plt.figure(figsize=(5, 3))
            plt.hist(errs, bins=40)
            plt.axvline(thr, color="r", ls="--",
                        label=f"{percentile:.0f}th pct = {thr:.4f}")
            plt.xlabel("reconstruction MSE (clean val)"); plt.ylabel("count")
            plt.title("Stage 1b: QC threshold calibration"); plt.legend(); plt.tight_layout(); plt.show()
        return thr, errs

    def evaluate_detector(model, tiles, is_artifact, threshold):
        """Optional supervised check when a few labelled tiles exist. The artifact
        (anomaly) is the POSITIVE class; PR-AUC is the honest headline metric because
        artifacts are rare (slide 42)."""
        errs = recon_errors(model, tiles)
        y = np.asarray(is_artifact).astype(int)
        pred_bad = (errs >= threshold).astype(int)
        tp = int(((pred_bad == 1) & (y == 1)).sum())
        fp = int(((pred_bad == 1) & (y == 0)).sum())
        fn = int(((pred_bad == 0) & (y == 1)).sum())
        out = {"precision": round(tp / max(tp + fp, 1), 3),
               "recall": round(tp / max(tp + fn, 1), 3)}
        if _HAVE_SK and len(set(y.tolist())) == 2:
            out["roc_auc"] = round(float(roc_auc_score(y, errs)), 3)
            out["pr_auc"] = round(float(average_precision_score(y, errs)), 3)
        return out


class SubTileQualityEvaluator:
    """Applies the calibrated gate across a full frame, tile by tile.
    flag = 1 => clean / pass (reconstruction error below threshold)."""

    def __init__(self, model=None, threshold_mse: float = 0.015, tile: int = 128):
        self.model = model
        self.threshold_mse = threshold_mse
        self.tile = tile

    def _tiles(self, img):
        t = self.tile
        for y in range(0, img.shape[0] - t + 1, t):
            for x in range(0, img.shape[1] - t + 1, t):
                yield y, x, img[y:y + t, x:x + t]

    def generate_quality_flags(self, image_array):
        if not (_HAVE_TORCH and self.model is not None):
            return []
        out = []
        for y, x, patch in self._tiles(image_array):
            mse = float(recon_errors(self.model, [patch])[0])
            out.append({"y": y, "x": x, "flag": int(mse < self.threshold_mse), "mse": mse})
        return out


# --------------------------------------------------------------------------
# DEMO: the FULL validated protocol on SYNTHETIC tiles (stand-in for HST cutouts).
# In real use, feed train_qc() CLEAN HST tiles and evaluate_detector() a small
# labelled set of clean vs known-bad frames. Replace _clean_tile/_artifact_tile.
# --------------------------------------------------------------------------
if _HAVE_TORCH:
    _rng = np.random.default_rng(0)

    def _clean_tile():
        yy, xx = np.mgrid[0:128, 0:128]
        cy, cx = _rng.uniform(40, 88, 2); s = _rng.uniform(6, 14)
        g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * s ** 2))
        return (g + _rng.normal(0, 0.02, (128, 128))).astype(np.float32)

    def _artifact_tile():
        t = _clean_tile()
        if _rng.random() < 0.5:
            t[_rng.integers(0, 128), _rng.integers(0, 128)] += _rng.uniform(5, 10)  # hot pixel
        else:
            r = _rng.integers(0, 126); t[r:r + 2, :] += _rng.uniform(2, 4)          # satellite streak
        return t

    torch.manual_seed(0)
    qc_model = QualityControlCAE(latent_dim=128)
    print("[Stage 1b] Training QC-CAE on CLEAN tiles (synthetic demo)...")
    train_qc(qc_model, [_clean_tile() for _ in range(160)], epochs=6, verbose=True)

    thr, _ = calibrate_threshold(qc_model, [_clean_tile() for _ in range(60)],
                                 percentile=99.0, plot=_HAVE_PLT)
    print(f"[Stage 1b] Calibrated threshold (99th pct of clean errors) = {thr:.5f}")

    _mix = [_clean_tile() for _ in range(60)] + [_artifact_tile() for _ in range(60)]
    _lab = [0] * 60 + [1] * 60
    print(f"[Stage 1b] Detector on held-out mix: {evaluate_detector(qc_model, _mix, _lab, thr)}")

    evaluator = SubTileQualityEvaluator(qc_model, threshold_mse=thr)
    print("[Stage 1b] Frame gate ready; latent_dim=128 also feeds Stage 1f.")
else:
    print("[Stage 1b] (Skipped - torch not available.)")


In [ ]:
# ============================================================
# Stage 1c — Detection, Deblending and Multi-Filter Alignment
# ============================================================
# Proposal (p.13): detect + deblend on a DESIGNATED detection band to get a master
# source list, cut 128x128 stamps at identical RA/Dec across all filters via absolute
# WCS, and MASK neighbouring sources so companions do not corrupt Stage 1e / statmorph.
import numpy as np

try:
    from astropy.wcs import WCS
    from astropy.nddata import Cutout2D
    from astropy.coordinates import SkyCoord
    import astropy.units as u
    from astropy.convolution import Gaussian2DKernel, convolve
    _HAVE_ASTROPY = True
except Exception as e:
    _HAVE_ASTROPY = False
    print(f"[Stage 1c] astropy unavailable ({e}); scaffold only.")

try:
    from photutils.background import Background2D, MedianBackground
    from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog
    _HAVE_PHOTUTILS = True
except Exception:
    _HAVE_PHOTUTILS = False


class MultiFilterAlignmentPipeline:
    def __init__(self, cutout_size: int = 128, detect_nsigma: float = 3.0,
                 npixels: int = 10):
        self.cutout_size = cutout_size
        self.detect_nsigma = detect_nsigma
        self.npixels = npixels
        self.segmap = None          # SegmentationImage from the detection band
        self.catalog = None         # SourceCatalog (master source list)

    def estimate_background_2d(self, image_data: np.ndarray):
        """Return (background-subtracted image, background RMS)."""
        if _HAVE_PHOTUTILS:
            bkg = Background2D(image_data, box_size=64, bkg_estimator=MedianBackground())
            return image_data - bkg.background, bkg.background_rms
        d = image_data.astype(np.float64)
        for _ in range(3):
            m, s = np.nanmedian(d), np.nanstd(d)
            d = np.where(np.abs(d - m) < 3 * s, d, np.nan)
        rms = np.nanstd(d)
        return image_data - np.nanmedian(d), np.full_like(image_data, rms)

    def detect_and_deblend(self, detection_image: np.ndarray):
        """DESIGNATED detection band -> master segmentation map + source catalogue.
        Run once on the deepest/combined band; all filters inherit these positions."""
        if not _HAVE_PHOTUTILS:
            print("[Stage 1c] photutils unavailable; detection skipped.")
            return None, None
        data, rms = self.estimate_background_2d(detection_image)
        conv = convolve(data, Gaussian2DKernel(2.0))
        segm = detect_sources(conv, self.detect_nsigma * rms, npixels=self.npixels)
        if segm is None:
            return None, None
        segm = deblend_sources(conv, segm, npixels=self.npixels, nlevels=32, contrast=0.001)
        self.segmap = segm
        self.catalog = SourceCatalog(data, segm)
        return segm, self.catalog

    def extract_masked_cutouts(self, target_label: int, position,
                               filter_images: dict, filter_wcs: dict) -> dict:
        """Cut every band at the same position; using the detection segmentation map,
        zero out pixels belonging to OTHER sources (neighbour masking)."""
        if self.segmap is None:
            raise RuntimeError("Run detect_and_deblend() on the detection band first.")
        size = (self.cutout_size, self.cutout_size)
        seg_arr = self.segmap.data.astype(int)
        aligned = {}
        for band, data in filter_images.items():
            try:
                cut = Cutout2D(data, position=position, size=size,
                               wcs=filter_wcs[band], mode="partial", fill_value=0.0)
                seg_cut = Cutout2D(seg_arr, position=position, size=size,
                                   mode="partial", fill_value=0).data
                out = cut.data.copy()                       # copy: never mutate the mosaic
                neighbour = (seg_cut != target_label) & (seg_cut != 0)
                out[neighbour] = 0.0                        # mask companion sources
                out[np.isnan(out)] = 0.0
                aligned[band] = out
            except Exception as e:
                print(f"[Stage 1c] alignment failure ({band}): {e}")
                aligned[band] = np.zeros(size, np.float32)
        return aligned


# --------------------------------------------------------------------------
# DEMO: target galaxy + a CLOSE companion + a far source; detect, then mask.
# --------------------------------------------------------------------------
if _HAVE_ASTROPY and _HAVE_PHOTUTILS:
    _rng = np.random.default_rng(0)
    yy, xx = np.mgrid[0:400, 0:400]
    def _g(cx, cy, amp, s): return amp * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * s ** 2))
    detection_band = (_g(200, 200, 50, 8) + _g(218, 200, 40, 5) + _g(320, 120, 45, 6)
                      + _rng.normal(0, 1.0, (400, 400))).astype(np.float32)

    w = WCS(naxis=2); w.wcs.crpix = [200, 200]; w.wcs.crval = [150.0, 2.0]
    w.wcs.cdelt = [-0.03 / 3600, 0.03 / 3600]; w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    filters = {"F606W": detection_band, "F814W": detection_band * 0.8}
    wcs_dict = {"F606W": w, "F814W": w}

    aligner = MultiFilterAlignmentPipeline(cutout_size=128)
    segm, cat = aligner.detect_and_deblend(detection_band)      # master source list
    xs, ys = cat.xcentroid, cat.ycentroid
    ti = int(np.argmin((xs - 200) ** 2 + (ys - 200) ** 2))       # target nearest (200,200)
    stamps = aligner.extract_masked_cutouts(cat.labels[ti], (xs[ti], ys[ti]), filters, wcs_dict)
    print(f"[Stage 1c] detected {len(cat)} sources; target label {cat.labels[ti]}; "
          f"channels {list(stamps.keys())}; neighbour masking applied.")
else:
    print("[Stage 1c] (Skipped demo - astropy/photutils not available; class is ready.)")


In [ ]:
# ============================================================
# Stage 1d — empirical PSF + CORRECT cross-filter homogenisation
# ============================================================
import numpy as np
from scipy.signal import fftconvolve

try:
    from photutils.psf.matching import create_matching_kernel, TopHatWindow
    _HAVE_PSFMATCH = True
except Exception:
    _HAVE_PSFMATCH = False


class CrossFilterPSFHomogeniser:
    """
    Homogenises resolution ACROSS bands. The correct operation is to build a
    matching kernel K with  psf_sharp (*) K ~= psf_broad  and convolve the
    SHARP band with K. Convolving an image directly with the broad PSF (the
    original code) blurs it PAST the target resolution.
    """

    def __init__(self, kernel_size: int = 25):
        self.kernel_size = kernel_size

    def extract_empirical_psf(self, star_cutouts) -> np.ndarray:
        """Stack several centred star cutouts into a normalised empirical PSF.
        Accepts one array or a list; real use should sub-pixel recentre and
        oversample each star before stacking."""
        if isinstance(star_cutouts, np.ndarray):
            star_cutouts = [star_cutouts]
        acc = None
        k = self.kernel_size
        for s in star_cutouts:
            c = s.shape[0] // 2
            h = k // 2
            core = s[c - h:c + h + 1, c - h:c + h + 1].astype(np.float64)
            acc = core if acc is None else acc + core
        tot = acc.sum()
        return acc / tot if tot > 0 else np.ones((k, k)) / (k * k)

    @staticmethod
    def _wiener_matching_kernel(psf_sharp, psf_broad, reg=1e-3):
        S = np.fft.fft2(np.fft.ifftshift(psf_sharp))
        B = np.fft.fft2(np.fft.ifftshift(psf_broad))
        K = B * np.conj(S) / (np.abs(S) ** 2 + reg)
        k = np.fft.fftshift(np.real(np.fft.ifft2(K)))
        return k / k.sum()

    def build_matching_kernel(self, psf_sharp, psf_broad):
        if _HAVE_PSFMATCH:
            return create_matching_kernel(psf_sharp, psf_broad, window=TopHatWindow(0.35))
        return self._wiener_matching_kernel(psf_sharp, psf_broad)

    def match_sharpness(self, sharp_image, psf_sharp, psf_broad) -> np.ndarray:
        """Degrade the sharp band to the broad band's resolution via K."""
        K = self.build_matching_kernel(psf_sharp, psf_broad)
        return fftconvolve(sharp_image, K, mode="same")


def _gauss_psf(size, sigma):
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    g = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return g / g.sum()


def _sigma_proxy(p):
    ax = np.arange(p.shape[0]) - p.shape[0] // 2
    xx, yy = np.meshgrid(ax, ax)
    w = np.clip(p, 0, None); w = w / w.sum()
    return float(np.sqrt(np.sum(w * (xx ** 2 + yy ** 2))))


hom = CrossFilterPSFHomogeniser()
psf_sharp = _gauss_psf(25, 1.5)                 # e.g. F814W
psf_broad = _gauss_psf(25, 2.5)                 # target (broader) resolution
img_sharp = _gauss_psf(128, 1.5)
matched = hom.match_sharpness(img_sharp, psf_sharp, psf_broad)
matched /= matched.sum()
print(f"[Stage 1d] sigma: sharp={_sigma_proxy(psf_sharp):.2f} target={_sigma_proxy(psf_broad):.2f} "
      f"matched={_sigma_proxy(matched):.2f}  (photutils kernel: {_HAVE_PSFMATCH})")


In [ ]:
# ============================================================
# Stage 1e — Pass-1 Photometry and Morphology (deterministic, no random values)
# ============================================================
# Computes Gini, Concentration, Asymmetry, Smoothness, M20, and a Sersic index
# deterministically (statmorph is used when installed). Every feature column is
# always populated so the tabular shape is stable across the NumPy fallback and the
# live statmorph path (M20 and Sersic n are real measurements, not fabricated).
import numpy as np
import pandas as pd

try:
    import statmorph
    from photutils.segmentation import detect_sources
    _HAVE_STATMORPH = True
except Exception:
    _HAVE_STATMORPH = False

try:
    from scipy.optimize import curve_fit
    from scipy.ndimage import uniform_filter
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False


class Pass1MorphologyEngine:
    @staticmethod
    def _segmap(img, nsigma=1.5):
        med = np.median(img)
        mad = np.median(np.abs(img - med)) * 1.4826 + 1e-8
        return img > (med + nsigma * mad)

    def gini(self, img, seg=None) -> float:
        pix = img[seg] if seg is not None else img.ravel()
        f = np.sort(np.abs(pix.ravel())); n = f.size; tot = f.sum()
        if n < 2 or tot == 0:
            return 0.0
        idx = np.arange(1, n + 1)
        return float(np.sum((2 * idx - n - 1) * f) / (tot * (n - 1)))

    def concentration(self, img) -> float:
        ny, nx = img.shape
        yy, xx = np.mgrid[0:ny, 0:nx]
        r = np.sqrt((xx - nx / 2) ** 2 + (yy - ny / 2) ** 2)
        order = np.argsort(r.ravel())
        cog = np.cumsum(np.clip(img.ravel()[order], 0, None))
        if cog[-1] <= 0:
            return 0.0
        cog = cog / cog[-1]; rs = r.ravel()[order]
        r20, r80 = rs[np.searchsorted(cog, 0.20)], rs[np.searchsorted(cog, 0.80)]
        return float(5.0 * np.log10(r80 / r20)) if r20 > 0 else 0.0

    def asymmetry(self, img) -> float:
        rot = img[::-1, ::-1]; denom = np.sum(np.abs(img))
        return float(np.sum(np.abs(img - rot)) / (2 * denom)) if denom > 0 else 0.0

    def smoothness(self, img, size=3) -> float:
        if not _HAVE_SCIPY:
            return 0.0
        sm = uniform_filter(img, size=size); denom = np.sum(np.abs(img))
        return float(np.sum(np.abs(img - sm)) / denom) if denom > 0 else 0.0

    def m20(self, img, seg=None) -> float:
        """Second-order moment of the brightest 20% of the flux (Lotz et al. 2004)."""
        I = np.clip(img, 0, None)
        if seg is not None:
            I = np.where(seg, I, 0.0)
        tot = I.sum()
        if tot <= 0:
            return np.nan
        ny, nx = I.shape; yy, xx = np.mgrid[0:ny, 0:nx]
        xc = (xx * I).sum() / tot; yc = (yy * I).sum() / tot
        d2 = (xx - xc) ** 2 + (yy - yc) ** 2
        Mtot = (I * d2).sum()
        if Mtot <= 0:
            return np.nan
        flat = I.ravel(); order = np.argsort(flat)[::-1]
        cum = np.cumsum(flat[order]); k = int(np.searchsorted(cum, 0.2 * tot))
        bright = np.zeros_like(flat); bright[order[:k + 1]] = flat[order[:k + 1]]
        M20 = (bright.reshape(I.shape) * d2).sum()
        return float(np.log10(max(M20, 1e-12) / Mtot))

    def sersic_n(self, img) -> float:
        """Rough Sersic index from a 1D azimuthally-averaged profile fit. A real
        measurement (bounded fit), not a placeholder; NaN if the fit fails."""
        if not _HAVE_SCIPY:
            return np.nan
        ny, nx = img.shape; yy, xx = np.mgrid[0:ny, 0:nx]
        r = np.sqrt((xx - nx / 2) ** 2 + (yy - ny / 2) ** 2).ravel()
        I = np.clip(img, 0, None).ravel()
        rb = np.linspace(0, nx / 2, 20); rr = 0.5 * (rb[:-1] + rb[1:])
        prof = np.array([I[(r >= rb[i]) & (r < rb[i + 1])].mean()
                         if ((r >= rb[i]) & (r < rb[i + 1])).any() else 0.0
                         for i in range(len(rb) - 1)])
        good = prof > 0
        if good.sum() < 4:
            return np.nan
        def _sersic(r, Ie, re, n):
            bn = 2 * n - 1 / 3.0
            return Ie * np.exp(-bn * ((r / max(re, 1e-3)) ** (1.0 / max(n, 0.3)) - 1))
        try:
            p, _ = curve_fit(_sersic, rr[good], prof[good],
                             p0=[prof[good].max(), nx / 8, 2.0],
                             bounds=([0, 1, 0.3], [np.inf, nx, 8.0]), maxfev=5000)
            return float(p[2])
        except Exception:
            return np.nan

    def evaluate_source(self, object_id: int, img: np.ndarray) -> pd.DataFrame:
        seg = self._segmap(img)
        rec = {
            "object_id": object_id,
            "mag_provisional": float(-2.5 * np.log10(np.sum(np.clip(img, 0, None)) + 1e-5)),
            "radius_pixels": float(np.sqrt(max(seg.sum(), 1))),
            "gini_index": self.gini(img, seg),
            "concentration": self.concentration(img),
            "asymmetry": self.asymmetry(img),
            "smoothness": self.smoothness(img),
            "m20_index": self.m20(img, seg),          # now a real value
            "sersic_n": self.sersic_n(img),           # now a real (rough) fit
            "n_source_pixels": int(seg.sum()),
            "flag": 0,
            "flag_sersic": 0,                          # placeholder; statmorph sets the real flag
            "feature_source": "statmorph" if _HAVE_STATMORPH else "numpy_fallback",
        }
        return pd.DataFrame([rec])


engine = Pass1MorphologyEngine()
ny = nx = 128
yy, xx = np.mgrid[0:ny, 0:nx]
_dev = np.exp(-7.67 * (((np.sqrt((xx - 64) ** 2 + (yy - 64) ** 2) / 12) ** 0.25) - 1))
df = engine.evaluate_source(339780, _dev)
print("[Stage 1e] Deterministic morphology row (M20 & Sersic populated):")
print(df.to_string(index=False))


## Phase 2 - Unsupervised Object Segregation

In [ ]:
# ============================================================
# Stage 2a — Representation Learning and Feature Fusion
# ============================================================
# Proposal spec: images asinh-scaled and encoded by a CAE into a 64-d latent;
# classical features log-transformed + RobustScaler; combined with PCA-reduced
# image vectors under explicit block-weighting; CAE/scaler/PCA fit on the TRAIN
# split only and applied by transform() to all held-out data.
import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    _HAVE_TORCH = True
except Exception as e:
    _HAVE_TORCH = False
    print(f"[Stage 2a] torch unavailable ({e}); scaffold only.")

from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

try:
    asinh_normalise                       # reuse Stage 1b's robust normaliser if present
except NameError:
    def asinh_normalise(patch, q_lo=1.0, q_hi=99.5, soft=0.1):
        p = patch.astype(np.float32)
        lo, hi = np.percentile(p, [q_lo, q_hi])
        x = np.clip((p - lo) / max(hi - lo, 1e-8), 0, 1)
        return (np.arcsinh(x / soft) / np.arcsinh(1.0 / soft)).astype(np.float32)

# Numeric classical-morphology columns emitted by Stage 1e.
MORPH_FEATURES = ["gini_index", "concentration", "asymmetry", "smoothness",
                  "radius_pixels", "mag_provisional", "m20_index", "sersic_n"]


def signed_log1p(x):
    """Sign-preserving log transform: safe for negative (m20) and zero values,
    so the proposal's 'log-transform the classical features' does not distort
    bounded or signed indices."""
    return np.sign(x) * np.log1p(np.abs(x))


if _HAVE_TORCH:
    class RepresentationCAE(nn.Module):
        """3-channel convolutional autoencoder. Conv 3->16->32->64 (128->16),
        Linear bottleneck to a 64-d latent, mirror decoder back to 3x128x128.
        Trained by reconstruction on the whole population; encode() returns the
        latent used by the fusion."""

        def __init__(self, in_ch=3, latent_dim=64):
            super().__init__()
            self.encoder_conv = nn.Sequential(
                nn.Conv2d(in_ch, 16, 3, stride=2, padding=1), nn.ReLU(),   # 128 -> 64
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),      # 64 -> 32
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),      # 32 -> 16
            )
            self._feat = 64 * 16 * 16
            self.enc_fc = nn.Linear(self._feat, latent_dim)
            self.dec_fc = nn.Linear(latent_dim, self._feat)
            self.decoder_conv = nn.Sequential(
                nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
                nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
                nn.ConvTranspose2d(16, in_ch, 3, 2, 1, output_padding=1), nn.Sigmoid(),
            )

        def encode(self, x):
            return self.enc_fc(self.encoder_conv(x).flatten(1))

        def decode(self, z):
            return self.decoder_conv(self.dec_fc(z).view(-1, 64, 16, 16))

        def forward(self, x):
            z = self.encode(x)
            return self.decode(z), z

    def _norm_stack(images):
        out = np.empty_like(images, dtype=np.float32)
        for n in range(images.shape[0]):
            for c in range(images.shape[1]):
                out[n, c] = asinh_normalise(images[n, c])   # asinh-scale each band
        return torch.tensor(out)

    def train_representation(model, images, epochs=15, batch_size=64,
                             lr=1e-3, weight_decay=1e-5, device="cpu", verbose=True):
        model.to(device).train()
        loader = DataLoader(TensorDataset(_norm_stack(images)),
                            batch_size=batch_size, shuffle=True)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        loss_fn = nn.MSELoss()
        for ep in range(epochs):
            run = 0.0
            for (xb,) in loader:
                xb = xb.to(device)
                recon, _ = model(xb)
                loss = loss_fn(recon, xb)
                opt.zero_grad(); loss.backward(); opt.step()
                run += loss.item() * xb.size(0)
            if verbose:
                print(f"  epoch {ep + 1:2d}/{epochs}  recon MSE = {run / len(loader.dataset):.5f}")
        return model

    @torch.no_grad()
    def encode_images(model, images, device="cpu"):
        model.to(device).eval()
        return model.encode(_norm_stack(images).to(device)).cpu().numpy()


class HybridFeatureFusion:
    """[PCA-reduced CAE latent  ||  log-transformed, RobustScaled classical features],
    each block whitened/robust-scaled to comparable scale and then block-weighted so
    neither dominates the clustering distance. fit on train, transform on held-out."""

    def __init__(self, encode_fn, n_pca=16, spatial_weight=0.5, feature_cols=None):
        self.encode_fn = encode_fn
        self.pca = PCA(n_components=n_pca, whiten=True)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        self.scaler = RobustScaler()
        self.spatial_weight = spatial_weight
        self.tabular_weight = 1.0 - spatial_weight
        self.feature_cols = list(feature_cols) if feature_cols else list(MORPH_FEATURES)
        self._kept_cols = None
        self.is_fitted = False

    def _tabular(self, df, fit):
        # FIXED schema: keep every configured column present in df (do NOT drop all-NaN
        # columns), so the matrix width is identical whether m20/sersic_n come from the
        # NumPy fallback or the live statmorph engine.
        if fit:
            self._kept_cols = [c for c in self.feature_cols if c in df.columns]
        X = df[self._kept_cols].to_numpy(dtype=float)
        # imputer fits ONLY when fit=True (no validation/test leakage); keep_empty_features
        # keeps any all-NaN column (filled with 0) instead of dropping it.
        X = self.imputer.fit_transform(X) if fit else self.imputer.transform(X)
        X = signed_log1p(X)                             # proposal: log-transform classical features
        return self.scaler.fit_transform(X) if fit else self.scaler.transform(X)

    def fit_transform(self, images, df):
        spat = self.pca.fit_transform(self.encode_fn(images))
        tab = self._tabular(df, fit=True)
        self.is_fitted = True
        return np.hstack([spat * self.spatial_weight, tab * self.tabular_weight])

    def transform(self, images, df):
        if not self.is_fitted:
            raise RuntimeError("Call fit_transform on the training split first.")
        spat = self.pca.transform(self.encode_fn(images))
        tab = self._tabular(df, fit=False)
        return np.hstack([spat * self.spatial_weight, tab * self.tabular_weight])


# --------------------------------------------------------------------------
# Synthetic demo data (3 populations). The tabular table is built with NumPy and
# is always available (Stage 2c uses it); only the CAE/fusion needs torch.
# --------------------------------------------------------------------------
def _make_demo(n_per=40, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:128, 0:128]
    def blob(s):
        g = np.exp(-((xx - 64) ** 2 + (yy - 64) ** 2) / (2 * s ** 2))
        return (np.stack([g * a for a in (1.0, 0.8, 0.6)]) + rng.normal(0, 0.02, (3, 128, 128))).astype(np.float32)
    specs = [("galaxy", 12.0, 3.0, 9.0, 0.55, 0.25, -18.0),
             ("star",    2.0, 4.5, 2.0, 0.65, 0.05, -16.0),
             ("noise",   0.5, 1.0, 1.0, 0.30, 0.02, -12.0)]
    imgs, rows, truth, oid = [], [], [], 0
    for name, s, conc, rad, gini, asym, mag in specs:
        for _ in range(n_per):
            imgs.append(blob(rng.normal(s, 1.0)))
            rows.append({"object_id": oid, "mag_provisional": float(rng.normal(mag, 0.3)),
                         "radius_pixels": float(abs(rng.normal(rad, 0.8))),
                         "gini_index": float(rng.normal(gini, 0.03)),
                         "concentration": float(rng.normal(conc, 0.2)),
                         "asymmetry": float(abs(rng.normal(asym, 0.02))),
                         "smoothness": float(abs(rng.normal(asym * 0.3, 0.01))),
                         "n_source_pixels": int(rad ** 2) + 1, "flag": 0,
                         "feature_source": "numpy_fallback",
                         "m20_index": np.nan, "sersic_n": np.nan,
                         "ra": 150.0 + rng.normal(0, 0.05), "dec": 2.2 + rng.normal(0, 0.05)})
            truth.append(name); oid += 1
    return np.stack(imgs), pd.DataFrame(rows), np.array(truth)


demo_images, demo_tabular_df, demo_truth = _make_demo()   # always available (NumPy)

if _HAVE_TORCH:
    torch.manual_seed(0)
    rep_cae = RepresentationCAE(in_ch=3, latent_dim=64)
    print("[Stage 2a] Training representation CAE on the full population (synthetic demo)...")
    train_representation(rep_cae, demo_images, epochs=5, verbose=True)
    fusion = HybridFeatureFusion(encode_fn=lambda im: encode_images(rep_cae, im),
                                 n_pca=16, spatial_weight=0.5)
    train_hybrid_features = fusion.fit_transform(demo_images, demo_tabular_df)
    print(f"[Stage 2a] Hybrid feature matrix: {train_hybrid_features.shape} "
          f"(kept tabular cols: {fusion._kept_cols})")
else:
    fusion = None
    train_hybrid_features = None
    print("[Stage 2a] (CAE/fusion demo skipped - torch not available; classes are ready.)")


In [ ]:
# ============================================================
# Stage 2b — Unsupervised Clustering
# ============================================================
# Proposal spec: a GMM (BIC sweep for the number of components) and HDBSCAN are
# fit on the TRAIN split; held-out objects are assigned via GMM.predict and
# HDBSCAN approximate_predict, NEVER by re-fitting. Output: soft posterior
# probabilities and hard cluster labels for every object.
# (No Gaia and no cluster naming here - that is Stage 2c/2d.)
import numpy as np
from sklearn.mixture import GaussianMixture

# Prefer the contrib `hdbscan` package (has approximate_predict for held-out
# inference); fall back to scikit-learn's HDBSCAN (fit-only) otherwise.
try:
    import hdbscan as _hdbscan_lib
    from hdbscan import HDBSCAN as _HDBSCAN
    _HDBSCAN_KIND = "contrib"
except Exception:
    try:
        from sklearn.cluster import HDBSCAN as _HDBSCAN
        _HDBSCAN_KIND = "sklearn"
    except Exception:
        _HDBSCAN = None
        _HDBSCAN_KIND = "none"


class UnsupervisedClustering:
    def __init__(self, k_range=(2, 6), hdbscan_min_cluster_size=25, random_state=42):
        self.k_range = k_range
        self.random_state = random_state
        self.gmm = None
        self.best_k = None
        if _HDBSCAN_KIND == "contrib":
            self.hdbscan = _HDBSCAN(min_cluster_size=hdbscan_min_cluster_size,
                                    prediction_data=True)     # enables approximate_predict
        elif _HDBSCAN_KIND == "sklearn":
            self.hdbscan = _HDBSCAN(min_cluster_size=hdbscan_min_cluster_size)
        else:
            self.hdbscan = None

    def _select_k_by_bic(self, X):
        best_bic, best = np.inf, None
        print("[Stage 2b] GMM BIC sweep:")
        for k in range(self.k_range[0], self.k_range[1] + 1):
            g = GaussianMixture(n_components=k, covariance_type="full",
                                n_init=10,                       # multi-start: avoid local minima
                                random_state=self.random_state).fit(X)
            bic = g.bic(X)
            print(f"   k={k}  BIC={bic:.1f}")
            if bic < best_bic:
                best_bic, best, self.best_k = bic, g, k
        print(f"[Stage 2b] BIC-selected k = {self.best_k}")
        return best

    def fit(self, X_train):
        """Fit GMM (BIC-selected k) and HDBSCAN on the training split."""
        self.gmm = self._select_k_by_bic(X_train)
        out = {"gmm_labels": self.gmm.predict(X_train),           # hard labels
               "gmm_probs": self.gmm.predict_proba(X_train),      # soft posteriors
               "best_k": self.best_k}
        if self.hdbscan is not None:
            out["hdbscan_labels"] = self.hdbscan.fit_predict(X_train)
        return out

    def predict(self, X_test):
        """Assign held-out objects WITHOUT re-fitting: GMM.predict for the GMM,
        and HDBSCAN.approximate_predict when the contrib package is available."""
        if self.gmm is None:
            raise RuntimeError("Call fit() on the training split first.")
        out = {"gmm_labels": self.gmm.predict(X_test),
               "gmm_probs": self.gmm.predict_proba(X_test)}
        if _HDBSCAN_KIND == "contrib" and self.hdbscan is not None:
            labels, strengths = _hdbscan_lib.approximate_predict(self.hdbscan, X_test)
            out["hdbscan_labels"] = labels
            out["hdbscan_strengths"] = strengths
        return out


# --------------------------------------------------------------------------
# DEMO (uses Stage 2a's hybrid feature matrix when torch produced it).
# --------------------------------------------------------------------------
if train_hybrid_features is not None:
    print(f"[Stage 2b] HDBSCAN backend: {_HDBSCAN_KIND}")
    clusterer = UnsupervisedClustering(k_range=(2, 6), hdbscan_min_cluster_size=25)
    res = clusterer.fit(train_hybrid_features)
    print(f"[Stage 2b] hard labels (first 8): {res['gmm_labels'][:8]}")
    print(f"[Stage 2b] soft posteriors (row 0): {np.round(res['gmm_probs'][0], 3)}")
    if "hdbscan_labels" in res:
        print(f"[Stage 2b] HDBSCAN clusters: {sorted(set(res['hdbscan_labels'].tolist()))}")
    held = clusterer.predict(train_hybrid_features[:5])          # demo held-out assignment
    print(f"[Stage 2b] held-out GMM.predict: {held['gmm_labels']}")
else:
    print("[Stage 2b] (Skipped demo - Stage 2a produced no features; class is ready.)")


In [ ]:
# ============================================================
# Stage 2c — Baseline Segregation
# ============================================================
# Proposal spec: a simple, transparent star / galaxy / noise separator built from
# (1) the size-magnitude locus, (2) a SExtractor CLASS_STAR criterion, and (3) a
# direct Gaia cross-match. Its labels feed the later ablation, quantifying the value
# the CAE-plus-clustering approach (Stage 2a-2b) adds over this baseline.
import numpy as np
import pandas as pd

try:
    from astroquery.gaia import Gaia
    from astropy.coordinates import SkyCoord
    import astropy.units as u
    _HAVE_GAIA = True
except Exception:
    _HAVE_GAIA = False


class BaselineSegregator:
    """Rule-based benchmark. Not learned - it is the yardstick the learned
    segregation must beat (Objective 2)."""

    def __init__(self, r_star_max=3.0, faint_mag=-13.0, noise_radius=1.5,
                 class_star_thresh=0.5, gaia_radius_arcsec=1.0):
        self.r_star_max = r_star_max          # stellar-locus size ceiling (pixels)
        self.faint_mag = faint_mag            # fainter than this + tiny -> noise
        self.noise_radius = noise_radius
        self.class_star_thresh = class_star_thresh
        self.gaia_radius_arcsec = gaia_radius_arcsec

    def class_star_proxy(self, df):
        """Stand-in for SExtractor's CLASS_STAR (0=galaxy .. 1=star). In production
        this column comes straight from Source Extractor; here it is approximated
        from compactness (small radius + high concentration -> star-like)."""
        conc = df["concentration"].to_numpy(dtype=float)
        rad = df["radius_pixels"].to_numpy(dtype=float)
        score = (conc - 1.0) / 4.0 + (self.r_star_max - rad) / self.r_star_max
        return 1.0 / (1.0 + np.exp(-3.0 * (score - 0.5)))     # squash to [0,1]

    def gaia_crossmatch(self, ra, dec):
        """Direct Gaia DR3 cone search per source; True where a Gaia source lies
        within the tolerance radius (a Gaia match is strong evidence of a star)."""
        if not _HAVE_GAIA:
            print("[Stage 2c] Gaia unavailable; baseline uses size-magnitude + CLASS_STAR only.")
            return None
        hits = np.zeros(len(ra), dtype=bool)
        for i, (r, d) in enumerate(zip(np.asarray(ra), np.asarray(dec))):
            j = Gaia.cone_search_async(SkyCoord(r, d, unit="deg"),
                                       radius=self.gaia_radius_arcsec * u.arcsec)
            hits[i] = len(j.get_results()) > 0
        return hits

    def segregate(self, df, use_gaia=True):
        """Return baseline Star / Galaxy / Noise labels + the intermediate signals."""
        cs = self.class_star_proxy(df)
        rad = df["radius_pixels"].to_numpy(dtype=float)
        mag = df["mag_provisional"].to_numpy(dtype=float)
        gaia_hits = (self.gaia_crossmatch(df["ra"], df["dec"])
                     if (use_gaia and "ra" in df.columns) else None)

        labels = np.empty(len(df), dtype=object)
        for i in range(len(df)):
            if gaia_hits is not None and gaia_hits[i]:
                labels[i] = "Star"                                    # direct Gaia match
            elif mag[i] > self.faint_mag and rad[i] < self.noise_radius:
                labels[i] = "Noise"                                   # faint + tiny -> spurious
            elif rad[i] < self.r_star_max and cs[i] > self.class_star_thresh:
                labels[i] = "Star"                                    # stellar locus + CLASS_STAR
            else:
                labels[i] = "Galaxy"
        return {"baseline_semantic": labels, "class_star": cs,
                "gaia_matched": gaia_hits}


# --------------------------------------------------------------------------
# DEMO (uses Stage 2a's synthetic feature table; runs with or without torch).
# --------------------------------------------------------------------------
if demo_tabular_df is not None:
    seg = BaselineSegregator()
    base = seg.segregate(demo_tabular_df, use_gaia=_HAVE_GAIA)
    labels = base["baseline_semantic"]
    from collections import Counter
    print("[Stage 2c] Baseline segregation vs synthetic truth:")
    for name in ["galaxy", "star", "noise"]:
        print(f"   true {name:7s}: {dict(Counter(labels[demo_truth == name]))}")
    print(f"[Stage 2c] Gaia leg active: {_HAVE_GAIA} (needs astroquery + network)")
else:
    print("[Stage 2c] (Skipped demo - no feature table available; class is ready.)")


## Notes for real-data runs

- **Synthetic demos** throughout so the code runs offline; swap in real inputs (clean tiles for 1b; the detection-band mosaic for 1c; the masked cutouts + Stage 1e table for 2a/2c).
- **Stage 1c** needs `photutils`/`astropy`; the detection band should be the deepest or combined filter. Neighbour masking zeroes companion pixels - for statmorph you can equivalently pass the target segment as `segmap` and companions as `mask`.
- **Stage 1e** M20 and the 1D Sersic fit are real but coarse; `statmorph` (used when installed) supersedes them and sets the true `flag`/`flag_sersic`.
- **Stage 2a** log-transform is sign-preserving (safe on m20); all-NaN columns are retained (filled with 0) until statmorph populates them.
- **Stage 2b** uses contrib `hdbscan` for `approximate_predict`; GMM handles held-out inference regardless.
- **Stage 2c** `class_star` is a compactness proxy for SExtractor CLASS_STAR; the Gaia leg needs astroquery + network.
